# Colab 01 — Safe setup and data staging

## Goal

Clone or fast-forward `v2-refactor`, install dependencies, mount Google Drive, and copy selected inputs into the temporary `/content` workspace.

**Nothing runs automatically.** Set one `RUN_*` switch at a time. The frozen `v1-final` tag is never checked out or modified.


## Setup


In [ ]:
from pathlib import Path
from importlib import import_module
import json
import os
import shutil
import subprocess
import sys

GIT_URL = "https://github.com/Thebluesky303/msc-dissertation-drug-repurposing.git"
BRANCH = "v2-refactor"
REPO_DIR = Path("/content/msc-dissertation-drug-repurposing")
WORKSPACE_DIR = Path("/content/dti_workspace")
DRIVE_ROOT = Path("/content/drive/MyDrive/msc-dissertation-drug-repurposing")


def configure_workspace_environment() -> None:
    # Configure all paths before importing config or any src module.
    os.environ["AML_GNN_PROJECT_ROOT"] = str(REPO_DIR)
    os.environ["AML_GNN_DATA_DIR"] = str(WORKSPACE_DIR / "data")
    os.environ["AML_GNN_MODELS_DIR"] = str(WORKSPACE_DIR / "checkpoints")
    os.environ["AML_GNN_RESULTS_DIR"] = str(WORKSPACE_DIR / "results")
    if str(REPO_DIR) not in sys.path:
        sys.path.insert(0, str(REPO_DIR))


def require_api(module_name: str, attribute: str):
    module = import_module(module_name)
    if not hasattr(module, attribute):
        raise RuntimeError(f"Missing required v2 API: {module_name}.{attribute}")
    return getattr(module, attribute)


def copy_directory(source: Path, destination: Path) -> None:
    if not source.exists():
        raise FileNotFoundError(source)
    destination.parent.mkdir(parents=True, exist_ok=True)
    shutil.copytree(source, destination, dirs_exist_ok=True)


print(json.dumps({
    "git_url": GIT_URL,
    "branch": BRANCH,
    "repo": str(REPO_DIR),
    "workspace": str(WORKSPACE_DIR),
    "drive_root": str(DRIVE_ROOT),
}, indent=2))


In [ ]:
RUN_CLONE_OR_UPDATE = False
RUN_INSTALL = False
RUN_MOUNT_DRIVE = False
RUN_STAGE_DATA = False


## Steps


In [ ]:
if RUN_CLONE_OR_UPDATE:
    if (REPO_DIR / ".git").is_dir():
        dirty = subprocess.check_output(
            ["git", "status", "--porcelain"], cwd=REPO_DIR, text=True
        ).strip()
        if dirty:
            raise RuntimeError("Refusing to update a dirty Colab checkout.")
        subprocess.run(["git", "fetch", "origin", BRANCH], cwd=REPO_DIR, check=True)
        subprocess.run(["git", "switch", BRANCH], cwd=REPO_DIR, check=True)
        subprocess.run(["git", "pull", "--ff-only", "origin", BRANCH], cwd=REPO_DIR, check=True)
    else:
        subprocess.run(
            ["git", "clone", "--branch", BRANCH, "--single-branch", GIT_URL, str(REPO_DIR)],
            check=True,
        )
    active = subprocess.check_output(
        ["git", "branch", "--show-current"], cwd=REPO_DIR, text=True
    ).strip()
    if active != BRANCH:
        raise RuntimeError(f"Expected {BRANCH}, found {active}.")
    print("Repository ready:", REPO_DIR)
else:
    print("Clone/update skipped: RUN_CLONE_OR_UPDATE is False.")


In [ ]:
if RUN_INSTALL:
    requirements = REPO_DIR / "requirements.txt"
    if not requirements.exists():
        raise FileNotFoundError(requirements)
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-r", str(requirements)],
        check=True,
    )
else:
    print("Dependency installation skipped: RUN_INSTALL is False.")


In [ ]:
if RUN_MOUNT_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
    print("Drive root:", DRIVE_ROOT)
else:
    print("Drive mount skipped: RUN_MOUNT_DRIVE is False.")


In [ ]:
if RUN_STAGE_DATA:
    copy_directory(DRIVE_ROOT / "data" / "raw", WORKSPACE_DIR / "data" / "raw")
    if (DRIVE_ROOT / "data" / "aml").exists():
        copy_directory(DRIVE_ROOT / "data" / "aml", WORKSPACE_DIR / "data" / "aml")
    if (REPO_DIR / "data" / "splits").exists():
        copy_directory(REPO_DIR / "data" / "splits", WORKSPACE_DIR / "data" / "splits")
    if (DRIVE_ROOT / "data" / "splits").exists():
        copy_directory(DRIVE_ROOT / "data" / "splits", WORKSPACE_DIR / "data" / "splits")
    print("Data staged under:", WORKSPACE_DIR / "data")
else:
    print("Data staging skipped: RUN_STAGE_DATA is False.")


## Checks

Confirm the branch is `v2-refactor`, inspect the staged filenames/checksums, and run `pytest -q` before training. Colab storage is temporary until explicitly synced to Drive.

## Next steps

Use Colab 02 for one baseline run or Colab 03 for one advanced run.
